# Phase 2.7 — Interaction diagnosis after failed replication

This notebook does not collect new model outputs. It diagnoses why the frozen scalar permutation law failed on Phase 2.6 and looks for a compact renderer-local interaction hypothesis.


## 1. Recover `/content` safely and clone the repo


In [ ]:
import os, shutil, subprocess
os.chdir('/content')
repo = '/content/proof-path-invariance'
if os.path.exists(repo):
    shutil.rmtree(repo)
subprocess.run(['git','clone','https://github.com/Kairose-master/proof-path-invariance.git',repo], check=True, cwd='/content')
os.chdir(repo)
print('cwd:', os.getcwd())


## 2. Generate and verify the frozen Phase 2.6 benchmark


In [ ]:
!python3 scripts/generate_s3_unseen_benchmark.py --out /tmp/s3_unseen_v0.jsonl
!python3 scripts/validate_s3_unseen_benchmark.py /tmp/s3_unseen_v0.jsonl
!python3 scripts/verify_s3_unseen_lock.py /tmp/s3_unseen_v0.jsonl


Continue only if the final line is `unseen-family benchmark lock verified`.


## 3. Upload the Phase 2.6 RAW DATA

Upload `pythia70m-step143000-s3-unseen-v0.jsonl`.


In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Upload exactly one Phase 2.6 raw JSONL file.')
name, data = next(iter(uploaded.items()))
raw_path = Path('/content') / name
raw_path.write_bytes(data)
print('uploaded:', raw_path, 'bytes:', raw_path.stat().st_size)


## 4. Re-run the confirmatory scorer as an integrity check


In [ ]:
!python3 scripts/score_s3_unseen_confirmatory.py "{raw_path}"


The primary result should still show `skill_vs_zero = -0.11878071118567624` and `success = false` before exploratory diagnosis.


## 5. Run Phase 2.7 interaction diagnosis


In [ ]:
!python3 scripts/analyze_s3_interactions.py --benchmark /tmp/s3_unseen_v0.jsonl --results "{raw_path}" | tee /content/phase2_7_interactions.json


## 6. Pairwise-order decomposition of each family


In [ ]:
!python3 scripts/decompose_s3_pairwise_order.py "{raw_path}" | tee /content/phase2_7_pairwise_order.json


Inspect each family's `r2`. A high value means its six S3 effects can be compressed by three pairwise precedence terms, but this is still family-specific and exploratory.


Inspect `grouped_effects`, `feature_effect_pearson`, and `phase2_vs_phase2_6`. This is exploratory only; do not treat a favorable pattern as confirmed.


## 7. Download interaction-analysis JSON

Run this cell by itself and wait for the download prompt to finish before running the next download cell.


In [ ]:
from google.colab import files
files.download('/content/phase2_7_interactions.json')


## 8. Download pairwise-order JSON

Run this as a separate cell so Colab opens a second download prompt independently.


In [ ]:
from google.colab import files
files.download('/content/phase2_7_pairwise_order.json')


Do not close Colab until **both** files are visible in your device's Downloads/Files app.
